# <img src="https://upload.wikimedia.org/wikipedia/commons/6/60/NISAR_artist_concept.jpg" width=400 align="left"/>
<img src="https://upload.wikimedia.org/wikipedia/commons/9/9b/NISAR_Mission_Logo.png" width=400 align="left"/><br><br><br><br><br>

# NISAR - bi-temporal Forest Disturbance Algorithm
### Forest Disturbance Algorithm: SAR Disturbance Product

Author: Josef Kellndorfer, Earth Big Data LLC (2022-02-03) <br>
Author: Adriana Parra, Leandra Sethares, Annemarie Peacock <br>

Objective: Use available simulated NISAR data (from ALOS2 images) to produce an example for ingesting NISAR into the ATBD algorithm. Because the ATBD algorithm requires a minimum of 18 months of NISAR time series of 12-day repeat datatakes, this notebook demonstrates core steps of the ATBD workflow with a bi-temporal change algorithm example.

For a demo of the LENO calval site, including a validation step where results are compared to pre-prepared classification data over the site, run the the NISAR_bitemporal_disturbance_demo_validation notebook.

### Summary

The workflow for generating a bi-temporal forest disturbance product from NISAR time series data stacks is implemented in executable python code in this notebook: 

- Selection of NISAR scenes based on the bi-temporal reference image from high-resolution optical data.
- Ingestion of NISAR data into a data stack (xarray)
- Application of the bi-temporal change detection algorithm (will be updated with the ATBD algorithm when time series data are available)
- Application of the 1 hectare scale validation approach

The simulated NISAR data stack from ALOS2 images was processed into GCOV products by Alex Christensen using the code available in the GitHub repository: https://github.com/achri19/NISAR_Prep_Workflows/

NOTE: Once NISAR GCOV data are available, the simulated data can be swapped with available NISAR data.

## Table of Contents
Step 1. [Set libraries and environment](#step-1)</br>
Step 2. [Get the NISAR data](#step-2)</br>
Step 3. [Bi-temporal Change Algorithm](#step-3)</br>
Step 4. [Generation of the NISAR disturbance product](#step-4)</br>
Step 5. [Saving Disturbance Products](#step-5)


## Step 1: Set libraries and environment <a class="anchor" id="step-1"></a>

Environment kernel: NISAR_Disturbance

Build environment from requirements.yml found in the [NISAR-Disturbance repository](https://github.com/NISAR-Science-Algorithms/NISAR_Disturbance)

1.1. [Import Libraries](#step-1-1)</br>

### Step 1.1: Import Libraries <a class="anchor" id="step-1-1"></a>

In [ ]:
import os,sys
from osgeo import gdal

import rioxarray

import rasterio
from rasterio.features import shapes
from rasterio.mask import mask
import rasterio.plot #new
from rasterio.plot import show
from rasterio.crs import CRS

# from matplotlib.lines import Line2D #new, to make legend
import matplotlib.pyplot as plt
from matplotlib import colors
import matplotlib.colors as mcolors

from scipy.ndimage import median_filter

# for Data access
import earthaccess

# Stream HDF5 in AWS
import h5py
import io
import re
import s3fs

# import requests
import datashader

# Standard modules (from atbd_disturbance)
import xarray as xr
import numpy as np
import pandas as pd
from datetime import datetime
import hvplot.xarray
import holoviews as hv

#PDF for validation report
import datetime
from datetime import datetime, date

# import itertools
from pyproj import CRS

import warnings
warnings.filterwarnings("ignore")

## Step 2: Get the NISAR Data<a class="anchor" id="step-2"></a>

2.1. [Access wasabi links](#step-2-1)</br>
2.2. [Inspect HDF5 File Contents](#step-2-2)</br>
2.3. [Read in HDF5 Files to a data stack](#step-2-3)</br>
2.4. [Define Study Area](#step-2-4)</br>

Will be replaced by data search of NISAR data (track/frame or bounding box search, ascending/descending)

### Step 2.1: Access data links <a class="anchor" id="step-2-1"></a>

The **wasabi_leno.txt** file contains a list of the bucket locations of publicly available simulated NISAR scenes over the LENO site. This file can be downloaded from the [NISAR-Disturbace github repository](https://github.com/NISAR-Science-Algorithms/NISAR_Disturbance). This will be used until data is publically available through ASF.

In [ ]:
## data will be accessed from a publicly available wasabi bucket otherwise users will define their own area
wasabi = True

In [ ]:
if wasabi:
    wasabi_reference_file = "../resources/wasabi_leno.txt"
    with open(wasabi_reference_file, 'r') as f:
        links = [x.removesuffix('\n') for x in f.readlines()]
else:
    #prompt user for box
    box_left = -120.833
    box_right = -116.3545
    box_bottom = 35.4498
    box_top = 32.4397
    
    auth = earthaccess.login()

    results = earthaccess.search_data(short_name = 'NISAR_L2_GCOV_', 
                                        # temporal = ('2023-07-01 00:00:00', '2023-08-31 23:59:59'), # can also specify by time
                                        bounding_box = (box_left, box_top, box_right, box_bottom )
                                       )
    ## Get URLS for search results
    links = [earthaccess.results.DataGranule.data_links(x, access ='direct')[0] for x in results if earthaccess.results.DataGranule.data_links(x, access ='direct')[0].endswith('.h5')]

## sort links by date
GCOV_dates = [pd.to_datetime(x.split('/')[-1].split('_')[11]) for x in links]
GCOV_dates, links = (list(t) for t in zip(*sorted(zip(GCOV_dates, links))))
  
all_GCOV_data = links    
print('There are %s results' % (len(all_GCOV_data)))

In [ ]:
## convert file list to list of observation times
date_to_times = {}

for file in [x.split('/')[-1] for x in all_GCOV_data]:
    parts = [ p for p in file.split('_') if 'T' in p ]
    for part in parts:
        # Check if part looks like 'YYYYMMDDTHHMMSS'
        if len(part) == 15 and part[8] == 'T':
            date = part[:8]
            time = part[9:]  # skip the 'T'
            if date not in date_to_times:
                date_to_times[date] = []
            date_to_times[date].append(time)
            break 

# Check if any dates have multiple observations at variable times
for date in date_to_times:
    unique_times = list(set(date_to_times[date]))
    if len(unique_times) > 1:
        print(f"Date {date} has multiple times: {sorted(unique_times)}")
    else:
        print(f"Date {date} has a single time: {unique_times[0]}")

Choose two available dates to compare.

In [ ]:
## add prompt
# start_date = "20220930"
# end_date = "20230918"
start_date = input("Start Date (YYYYMMDD): ")
end_date = input("End Date (YYYYMMDD): ")

In [ ]:
#filter dates based on start_date, end_date
sdate_obj = pd.to_datetime(start_date)
edate_obj = pd.to_datetime(end_date)

indices = []
temp = [abs(x - sdate_obj) for x in GCOV_dates]
indices.append([int(i) for i in np.arange(len(temp)) if temp[i]==min(temp)][0])
temp = [abs(x - edate_obj) for x in GCOV_dates]
indices.append([int(i) for i in np.arange(len(temp)) if temp[i]==min(temp)][0])

In [ ]:
## filter 
print("Dates of observation (HV):")
date_array = []
SAR_images = []
for ii in indices:
    datestr = all_GCOV_data[ii].split('/')[-1].split('_')[11]
    date_obj= pd.to_datetime(datestr)
    print(date_obj)
    date_array.append(date_obj)
    SAR_images.append(all_GCOV_data[ii])
# auth

### Step 2.2: Inspect HDF5 File Contents <a class="anchor" id="step-2-2"></a>

In [ ]:
def open_h5_file(filename, wasabi=True):
    try:
        f = h5py.File(s3.open(filename, "rb"), "r") 
    except:
        if wasabi:
            endpoint = "https://s3.us-west-1.wasabisys.com"
            fs = s3fs.S3FileSystem(anon=True, client_kwargs={"endpoint_url": endpoint, "region_name": "us-west-1"})  # Set to True for anonymous, public buckets
        else:
            fs = s3fs.S3FileSystem(anon=False)
        f = h5py.File(fs.open(filename, 'rb'), 'r')
        
    return f

##### Select a reference image to get dimension information

In [ ]:
ref_image= SAR_images[1]

f = open_h5_file(ref_image, wasabi=wasabi)
 
a_group_key = list(f.keys())[0]
ds_x = f[a_group_key]['LSAR']['GCOV']['grids']['frequencyA']['xCoordinates'][()]      # returns as a h5py dataset object
ds_y = f[a_group_key]['LSAR']['GCOV']['grids']['frequencyA']['yCoordinates'][()]      # returns as a h5py dataset object
ds_epsg = f[a_group_key]['LSAR']['GCOV']['grids']['frequencyA']['projection'][()]
mask = np.where(np.isnan(f[a_group_key]['LSAR']['GCOV']['grids']['frequencyA']['HVHV'][()]),np.nan,1)

x_coordinates_spacing = f[a_group_key]['LSAR']['GCOV']['grids']['frequencyA']['xCoordinateSpacing'][()]
y_coordinates_spacing = f[a_group_key]['LSAR']['GCOV']['grids']['frequencyA']['yCoordinateSpacing'][()]

cols = ds_x.shape[0]
rows = ds_y.shape[0]
print('X Size: ',cols,' Y Size: ',rows)

yres = abs(ds_y[0] - ds_y[1])
xres = abs(ds_x[0] - ds_x[1])
print('Resolution X:', xres, ' Y:',yres,'m')

ulx = ds_x[0] - ((ds_x[1] - ds_x[0])/2)
lrx = ds_x[-1] + ((ds_x[1] - ds_x[0])/2)
uly = ds_y[0] - ((ds_y[1] - ds_y[0])/2)
lry = ds_y[-1]+ ((ds_y[1] - ds_y[0])/2)

print('Raster bounds: ',ulx,lrx,uly,lry)#min(ds_x),max(ds_x),min(ds_y),max(ds_y))
geotransform = [ulx, xres, 0.0, uly, 0.0, -yres]

spatialref = CRS.from_epsg(ds_epsg).to_wkt()
print(ds_epsg)

In [ ]:
# plot the reference image
# Retrieve the data array from the h5py file object
HVHV_data = f[a_group_key]['LSAR']['GCOV']['grids']['frequencyA']['HVHV'][()]
extent = [ulx, lrx, lry, uly]

# Create the plot
plt.figure(figsize=(7, 5))
plt.imshow(HVHV_data, cmap='gray', extent=extent, vmin=0, vmax=0.2)
plt.title(f'HVHV Polarization of SAR Image\nEPSG: {ds_epsg}')
plt.colorbar(label='HVHV Backscatter', shrink=0.9)
plt.show()

### Step 2.3: Read in HDF5 Files to a data stack <a class="anchor" id="step-2-3"></a>

In [ ]:
# List to store filenames of images that pass the dimension check
filenames = []

# Read raster files and make them into a 3D numpy array
arrs_HV = []

for image in SAR_images:
    filename = os.path.split(image)[-1]
    print(filename)
    f = open_h5_file(image, wasabi=wasabi)
    
    a_group_key = list(f.keys())[0]
    ds_HV = f[a_group_key]['LSAR']['GCOV']['grids']['frequencyA']['HVHV'][()]
    input_values = ds_HV.ravel()
    dataset = np.where(np.isinf(ds_HV), np.nan, ds_HV)

    if (ds_HV.shape[1] == cols) & (ds_HV.shape[0] == rows):
        arrs_HV.append(dataset)
        filenames.append(filename)  # Save the filename of the valid image
    else:
        print(f'Dimensions of this SAR image do not match the reference image, SKIP')
        del ds_HV, f

# Convert the list of valid images into a numpy array
a_HV = np.array(arrs_HV, dtype=float)

# Create a variable for the number of images (one for each date)
num_dates = len(arrs_HV)
print("\nNumber of dates used in time series analysis:", num_dates)

del arrs_HV #delete to save space

##### To preview the data, plot the images

In [ ]:
cmp = plt.get_cmap('Greys_r')
plt.rcParams['figure.figsize'] = (20,15)
for i in range(0,2):
    ax = plt.subplot(2,2,i+1)
    plt.imshow(a_HV[i,:,:],vmin=0,vmax=0.2,cmap=cmp)
    plt.colorbar(label = 'HV', shrink=0.9)
    ax.axes.get_xaxis().set_ticks([])
    ax.axes.get_yaxis().set_ticks([])
    plt.title(date_array[i],fontsize=16) 
    plt.tight_layout()

##### Get the acquisition date from the file name and create a list of dates.

In [ ]:
# Define a function to extract date from the filename
def extract_date_from_filename(filename):
    match = re.search(r'\d{8}T\d{6}', filename) #set to just d{8} if just getting YYYYMMDD
    return match.group(0) if match else None
    
# Extract dates from filenames
date_format = '%Y%m%dT%H%M%S' #Use the time in the file saving step
dates = []

file_list = filenames
for file in file_list:
    filename = os.path.basename(file)
    date_str = extract_date_from_filename(filename)
    date = datetime.strptime(date_str, date_format)
    dates.append(date)
    
# Create a Pandas DatetimeIndex
dates = pd.DatetimeIndex(dates)
dates

##### Create an xarray from the data stack, ensuring all images have the same dimensions

In [ ]:
# array shape
print(a_HV.shape, 'a_HV dimensions')

# Spatial dimensions
y_coordinates = ds_y 
x_coordinates = ds_x 

# Check for duplicate times and remove them
# Simulate times for this scenario (if not already provided)
unique_times, unique_indices = (
    pd.Series(dates).drop_duplicates().reset_index(drop=True),
    pd.Series(dates).drop_duplicates().index)

# Extract filtered data based on unique times
filtered_data_arrays = [a_HV[i, :, :] for i in unique_indices]
# Ensure all arrays have the same shape
array_shapes = [arr.shape for arr in filtered_data_arrays]
print("Original shapes of data arrays:", array_shapes)
# Determine the target shape for alignment
target_shape = max(array_shapes, key=lambda x: (x[0], x[1]))
# Adjust arrays to match the target shape
adjusted_arrays = []
for arr in filtered_data_arrays:
    if arr.shape != target_shape:
        # Resize or pad `arr` to match `target_shape`
        padded_arr = np.zeros(target_shape)
        padded_arr[: arr.shape[0], : arr.shape[1]] = arr
        adjusted_arrays.append(padded_arr)
    else:
        adjusted_arrays.append(arr)

# Verify the adjusted shapes
array_shapes = [arr.shape for arr in adjusted_arrays]
print("Adjusted shapes of data arrays:", array_shapes)

# Create an xarray.DataArray
gcov_xarray = xr.DataArray(
    data=np.stack(adjusted_arrays, axis=0),  # Stack along a new time dimension
    dims=("time", "y", "x"),
    coords={
        "time": unique_times,
        "y": y_coordinates,
        "x": x_coordinates,
    },
    name="GCOV_data",
)
# Convert DataArray to Dataset
gcov_dataset = gcov_xarray.to_dataset()

# Set xarray Dataset attributes
gcov_dataset.attrs = {
    "crs": CRS.from_epsg(ds_epsg),
    "nodatavals": [0.0], 
    "res": [x_coordinates_spacing, y_coordinates_spacing],   # Resolution for x and y
}
gcov_dataset.rio.write_crs(CRS.from_epsg(ds_epsg), inplace=True)
gcov_dataset

# Reorder dimensions to ('time', 'y', 'x')
gcov_dataset_transposed = gcov_dataset.transpose('time', 'y', 'x')
gcov_dataset_transposed

### Step 2.4: Define Study Area <a class="anchor" id="step-2-4"></a>

##### Subset NISAR image to a smaller area with potential forest disturbance
Reasons to subset the NISAR scene include: calibration error, coastline, part of the image is non-forested, etc. In this case we will subset to the same area our validation data covers

##### Define smaller extent

In [ ]:
print('Scene Extent: ',ulx,lrx,uly,lry)#min(ds_x),max(ds_x),min(ds_y),max(ds_y))
geotransform = [ulx, xres, 0.0, uly, 0.0, -yres]

print("Set a smaller extent to study")
### LENO default to minx, maxx, miny, maxy = 378225.0, 399225.0, 3548756.0, 3568056.0
[minx, maxx] = [float(x) for x in input("min(x), max(x):").split(',')]
[miny, maxy] = [float(y) for y in input("min(y), max(y):").split(',')]

print('minx, maxx, miny, maxy:', minx, maxx, miny, maxy)

# Define a subset with change
xslice = [minx, maxx]
yslice = [miny, maxy]

xslice.sort()
yslice.sort(reverse=True)

In [ ]:
ds_s = gcov_dataset.sel({'x':slice(*xslice),'y':slice(*yslice)})
ds_s

## Step 3: Bi-temporal Change Algorithm <a class="anchor" id="step-3"></a>

3.0. [Preview data](#step-3-0)</br>
3.1. [Apply Median Filter](#step-3-1)</br>
3.2. [Normalized Radar Change Index](#step-3-2)</br>
3.3. [HV Log Ration](#step-3-3)</br>
3.4. [Generate inital Disturbance Product](#step-3-4)</br>

### Step 3.0: Preview Data  <a class="anchor" id="step-3-0"></a>

##### View the NISAR images before starting the Bi-temporal Change Algorithm

In [ ]:
# Read in two dates
t0 = ds_s["GCOV_data"][0] #first date 
t1 = ds_s["GCOV_data"][1] #second date

In [ ]:
# Plot t0 and t1
fig, [ax1, ax2] = plt.subplots(dpi=80, ncols=2)

date_t0 = dates[0].date().strftime('%Y-%m-%d')
date_t1 = dates[1].date().strftime('%Y-%m-%d')

# Plot 1: t0
pos1 = ax1.imshow(t0, cmap='Greys_r',vmin=0,vmax=0.1)#, interpolation='nearest')
ax1.set_title(f'HV t0, {date_t0}')
cbar1 = fig.colorbar(pos1, ax=ax1, fraction=0.04, pad=0.04)
cbar1.set_label('HV')

# Plot 2: t1
pos2 = ax2.imshow(t1, cmap='Greys_r',vmin=0,vmax=0.1)#, interpolation='nearest')
ax2.set_title(f"HV t1, {date_t1}")
cbar2 = fig.colorbar(pos2, ax=ax2, fraction=0.04, pad=0.04)
cbar2.set_label('HV')

plt.tight_layout()
plt.show()

##### Make a simple forest mask by thresholding dB

Currently not applied to data. For later use in time series-based algorithm. The mask will be applied to a reference baseline time series.

In [ ]:
# Convert Linear to dB
def linear_to_db(sigma_linear):
    return 10 * np.log10(sigma_linear)
db_val = linear_to_db(t0)

forest_mask_thresh = -20

# Forest / Non-forest classification
forest_mask = db_val > forest_mask_thresh  # threshold in dB

# Plot
fig, ax = plt.subplots(figsize=(10, 10))
cax = ax.imshow(forest_mask, cmap='Greens', interpolation='none')
ax.set_title(f"Forested Areas above {forest_mask_thresh} dB HV")
cbar = fig.colorbar(cax, ax=ax, fraction=0.046, pad=0.04)
cbar.set_ticks([0, 1])
cbar.set_ticklabels(["Not Forested", "Forested"])
plt.show()

### Step 3.1: Apply Median Filter to data <a class="anchor" id="step-3-1"></a>

Apply a median filter with a 5x5 window over the data to decrease noise

In [ ]:
from scipy.ndimage import median_filter

def apply_median_filter(data_array, x, y):
    # Apply window filter over spatial dimensions (x, y)
    return median_filter(data_array, size=(x, y))

#set window size
x = 5
y = 5

# Apply median filter
t0_f = apply_median_filter(t0, x, y) 
t1_f = apply_median_filter(t1, x, y) 

### Step 3.2: Normalized Radar Change Index <a class="anchor" id="step-3-2"></a>

The ***Normalized Radar Change Index*** uses bi-temporal HV data. In the HV normalized difference, values should range between -1 and 1: 
* High values: HV backscatter increased from t0 to t1
* Low values: HV backscatter decreased from t0 to t1, suggesting forest disturbance
* Close to 0: little to no change in cross-polarized backscatter between dates


\begin{align}
\text{NRCI} = \frac{HV_{t1} - HV_{t0}}{HV_{t1} + HV_{t0}}
\end{align}

$HV_{t0}$  -  first HV image <br>
$HV_{t1}$  -  second HV image <br>
  

In [ ]:
# NRCI of unfiltered data
nrci = (t1 - t0) / (t1 + t0)

# NRCI with median filter applied
nrci_f = (t1_f - t0_f) / (t1_f + t0_f)

In [ ]:
fig, [ax1, ax2] = plt.subplots(dpi=80, ncols=2)

# Plot 1: NRCI
pos1 = ax1.imshow(nrci, cmap='Greys_r', vmin=-1, vmax=1)
ax1.set_title('NRCI')
cbar1 = fig.colorbar(pos1, ax=ax1, fraction=0.04, pad=0.04)
cbar1.set_label('NRCI')

# Plot 2: NRCI Filtered
pos2 = ax2.imshow(nrci_f, cmap='Greys_r', vmin=-1, vmax=1)
ax2.set_title(f"NRCI with Median Filter, window size ({x}, {y})")
cbar2 = fig.colorbar(pos2, ax=ax2, fraction=0.04, pad=0.04)
cbar2.set_label('NRCI')

plt.tight_layout()
plt.show()

### Step 3.3: HV Log Ratio <a class="anchor" id="step-3-3"></a>

The HV log ratio measures the logarithmic change in cross-polarized HV backscatter values between two dates. Because HV backscatter is strongly influenced by volume scattering linked to forest structure and biomass, this ratio is a useful indicator of forest disturbance. A decrease in HV typically reflects a loss of canopy cover and a reduction in vegetation volume.

\begin{align}
\text{HV log ratio} = {log}\frac{HV_{t1}}{HV_{t0}}
\end{align}

$HV_{t0}$  -  first HV image <br>
$HV_{t1}$  -  second HV image <br>

* High values: HV backscatter increased from t0 to t1
* Low values: HV backscatter decreased from t0 to t1, suggesting forest disturbance
* Close to 0: little to no change in cross-polarized backscatter between dates

In [ ]:
# Calculate HV Log Ratio
hv_log = np.log(t1/t0)
# Apply median filter
hv_log_f = np.log(t1_f/t0_f)

In [ ]:
fig, [ax1, ax2] = plt.subplots(dpi=80, ncols=2)

# Plot 1: HV Log Ratio
pos1 = ax1.imshow(hv_log, cmap='Greys_r', vmin=-3, vmax=3)
ax1.set_title('HV Log Ratio')
cbar1 = fig.colorbar(pos1, ax=ax1, fraction=0.04, pad=0.04)
cbar1.set_label('HV Log Ratio')

# Plot 2: HV Log Filtered
pos2 = ax2.imshow(hv_log_f, cmap='Greys_r', vmin=-4, vmax=4)
ax2.set_title(f"HV Log Ratio with Median Filter, window size ({x}, {y})")
cbar2 = fig.colorbar(pos2, ax=ax2, fraction=0.04, pad=0.04)
cbar2.set_label('HV Log Ratio')

plt.tight_layout()
plt.show()

### Step 3.4: Generate inital Disturbance Product <a class="anchor" id="step-3-4"></a>

Apply thresholding to bi-temporal change results to get an inital 20m binary distrubance product

In [ ]:
# Convert back to xarray to make interactive hvplot to inspect for threshold values
nrci_da = np.array(nrci_f, dtype=np.float32)  # convert from object to float
ny, nx = nrci_da.shape
nrci_xr = xr.DataArray(nrci_da,dims=("y", "x"), coords={"y": np.arange(ny), "x": np.arange(nx)})

hv_log_da = np.array(hv_log_f, dtype=np.float32)  # convert from object to float
hv_log_xr = xr.DataArray(hv_log_da,dims=("y", "x"), coords={"y": np.arange(ny), "x": np.arange(nx)})

Generate an interactive plot to determine an appropriate disturbance threshold

In [ ]:
# Plot
hvplot.extension('bokeh') #needed for plots to generate 

nrci_clim = (-1,1) #set for visualization 
nrci_xr = nrci_xr.assign_coords(y=np.arange(ny-1, -1, -1)) # otherwise the plot is flipped
nrci_hvp=nrci_xr.hvplot.image(x="x",y="y",rasterize=True,cmap='Greys_r', frame_width=600,frame_height=600,clim=nrci_clim,title='NRCI (Filtered)') 

hv_log_clim = (-2,2)
hv_log_xr = hv_log_xr.assign_coords(y=np.arange(ny-1, -1, -1)) # otherwise the plot is flipped
hv_log_hvp=hv_log_xr.hvplot.image(x="x",y="y",rasterize=True,cmap='Greys_r', frame_width=600,frame_height=600,clim=hv_log_clim,title='HV Log-Ratio (Filtered)') 

nrci_hvp + hv_log_hvp

In [ ]:
# Apply threshold to identify areas of disturbance
dthres = -0.3
dist_20m = nrci_f < dthres

In [ ]:
# Create colormap and norm
labels = ["Not Disturbed","Disturbed"]
colors = ["white", "black"] 
cmap = mcolors.ListedColormap(colors)
norm = mcolors.BoundaryNorm([0, 1, 2], cmap.N)

# Plot
fig, ax = plt.subplots(dpi=90)
pos=ax.imshow(dist_20m, cmap=cmap)
cbar = fig.colorbar(pos, ax=ax, ticks=[0.25, 0.75])
cbar.ax.set_yticklabels(labels)
ax.set_title(f'Disturbance Product - Disturbed Areas where NRCI < {dthres} threshold', fontsize=16)
plt.show()

## Step 4: Generation of the NISAR Disturbance Product <a class="anchor" id="step-4"></a>

We are coarsening the data set to one-hectare cells by applying a count in 5x5 blocks (to get 100x100 sqm from the 20x20 sqm pixels). An intermediate product is the count of 20 m pixels that were detected to change inside a one-hectare pixel. This product is thresholded by a minimum count of five pixels to classify a one-hectare pixel as disturbed. The threshold is applied to remove classification noise and can be adapted.   

4.1. [Prepare Data for NISAR Disturbance Product](#step-4-1) </br>
4.2. [Plot count of disturbed pixels count in one-hectare cells](#step-4-2) </br>
4.3. [Plot 100 meter Disturbance Product](#step-4-3)</br>



### Step 4.1: Prepare data for the nisar disturbance products <a class="anchor" id="step-4-1"></a>

We coarsen the 20 m resolution data to one-hectare cells and count how many pixels we encounter in each one-hectare cell. By thresholding these 'count' image with the minimum required number of pixels to classify a cell disturbed at hectare scale, we can generate the final NISAR disturbance product. This procedure is done for each year of analysis.


For >50% forest cover change, would be >12.5 pixels

In [ ]:
# To run coarsen function, convert thresholded disturbance map from a numpy array to an xarray DataArray
dist_20m = np.array(dist_20m, dtype=np.float32)  # convert from object to float
dist_da = xr.DataArray(dist_20m, dims=("x", "y"))

# DEFAULT PIXEL THRESHOLD SET TO 5
pixelthres = 5
print('Pixel threshold for number of 20 meters pixels changed per 100 meters: ', pixelthres)

# Now, can coarsen the 20 meter NISAR product to 100 meters with the following function
# Function to get binary changed/not-changed maps in 20m and count of changed pixels per 100m
def find_binary_change(dist_20, pixelthres=pixelthres):

    # Get count of 20 meters change pixels in 100 meters (1 ha)
    # boundar='exact' throws error if not divisible by 5, options for ='pad' or ='trim'
    dist_100 = dist_20.coarsen(x=5, y=5, boundary='pad',coord_func='median').sum()
    dist_100_count = xr.where(dist_100 > 0, dist_100, 0)

    # Coarsen 20m to 100m and find where more than 5 pixels changed
    dist_100 = xr.where(dist_100 >= pixelthres, 1, 0)

    return dist_20, dist_100_count, dist_100

# Run function
dist_20m, dist_100m_count, dist_100m = find_binary_change(dist_da)

### Step 4.2: Plot count disturbed pixels count in one-hectare or 100 m cells<a class="anchor" id="step-4-2"></a>

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(dist_100m_count, cmap='magma', vmin=np.nanmin(dist_100m_count), vmax=np.nanmax(dist_100m_count))#, interpolation='nearest')
ax.set_title('Disturbance pixel count within 100 meters')
cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04, )
cbar.set_label('Disturbance Pixel Count')

 Figure: Count of disturbed 20 m pixels in one-hectare cells. 

### Step 4.3: Plot 100 meter Disturbance Product <a class="anchor" id="step-4-3"></a>

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(dist_100m, cmap='Grays', vmin=0, vmax=1)#, interpolation='nearest')
ax.set_title(f'Disturbance Product (100 meter resolution)\nwhere {pixelthres} or more pixels were disturbed in 100 meters')
cbar = plt.colorbar(pos, ax=ax, ticks=[0.25, 0.75], shrink=0.6)
cbar.ax.set_yticklabels(labels)

Figure: NISAR disturbance products at 100 m, from thresholding the 100 meter count of disturbed 20 m products. 

In [ ]:
## use a 100m calssification file to remove pixels where the validation data is not defined
if wasabi:
    mask_image = f"/vsicurl/https://s3.us-west-1.wasabisys.com/{CLS_file}"
else:
    mask_image = CLS_file
mask_ds = rasterio.open(mask_image).read(1)

dist_100m_m = dist_100m.where(mask_ds != 0)

In [ ]:
# Set colorbar
labels = ["Not Disturbed","Disturbed"]
colors = ["grey", "black"] 
cmap = mcolors.ListedColormap(colors)

fig, [ax1, ax2] = plt.subplots(dpi=90, ncols=2)

# Plot NISAR Disturbance
pos=ax1.imshow(dist_100m, cmap=cmap, interpolation='nearest')
ax1.set_title('NISAR Disturbance (100 meters)')
cbar1 = fig.colorbar(pos, ax=ax1, ticks=[0.25, 0.75], shrink=0.4)
cbar1.ax.set_yticklabels(labels)

# Plot Validation Disturbance
pos2=ax2.imshow(dist_100m_m, cmap=cmap, interpolation='nearest')
ax2.set_title("NISAR Disturbance - masked (100 meters)")
cbar2 = fig.colorbar(pos, ax=ax2, ticks=[0.25, 0.75], shrink=0.4)
cbar2.ax.set_yticklabels(labels)

## Step 5: Saving Disturbance Products <a class="anchor" id="step-5"></a>

Save COG of the 100m count product and 20m and 100m binary bitemporal disturbance products in a new disturbance_output directory

In [ ]:
sitename = input("Name disturbance area:")

In [ ]:
outdir = '../disturbance_output'
basename = f"NISAR_bitemporal_disturbance_{sitename}_{start_date}_{end_date}"
os.makedirs(outdir, exist_ok=True)

In [ ]:
out_100   = f"{outdir}/{basename}_100m.tif"
out_20    = f"{outdir}/{basename}_20m.tif"
out_count = f"{outdir}/{basename}_100m_count.tif"

for file, data in zip([out_100, out_20, out_count], [dist_100m, dist_20m, dist_100m_count]):
    print(f"Writing to {file}")
    tmpfile = f"{outdir}/dist_tmp.tif"
    data.transpose('y', 'x').rio.to_raster(tmpfile)
    ## convert to COG
    out = gdal.Translate(file, tmpfile, format="COG")
    os.remove(tmpfile)